<a href="https://colab.research.google.com/github/Bormey-Sky/Autoregressive-and-Diffusion-Language-Models/blob/main/AR_vs_Diffusion_Experiments_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AR vs Diffusion Language Models — Experiments
### Paper: A Comparison of Autoregressive and Diffusion Models in Sentence Generation
**Author:** Bormey Chanchem (Matr. 1842912) — Trier University

---

**Variable naming convention (consistent throughout):**
- `llama_model`, `llama_tok` — LLaMA-3-8B-Base (AR)
- `llada_model`, `llada_tok`, `MASK_ID` — LLaDA-8B-Instruct (Diffusion)

**Experiments:**
1. Reversal Reasoning — fine-tune on forward-only causal facts, test both directions
2. Contextual Infilling — restore missing spans from WikiText-2 sentences

**Session workflow:**
- **LLaMA-3 session:** Setup → 2.1 → 3.1 → 4 (IS_LLADA=False) → 7-9
- **LLaDA session:** Setup → 2.2 → 3.2 → 4 (IS_LLADA=True) → 7-9

---
## Section 0 — Setup
Run once at the start of every session.

In [1]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU detected.')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Sun Mar 29 22:20:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [15]:
%%capture
# LLaMA-3 session: transformers latest works fine
# LLaDA session: run !pip install transformers==4.38.2 -q after this cell
!pip install transformers accelerate bert-score -q
print('Dependencies installed.')

In [23]:
# LLaDA session: run
!pip install transformers==4.38.2 accelerate bert-score -q

In [2]:
import torch, json, os, random, sys, types, importlib.util, math
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

def show_table(rows, title=''):
    if not rows: return
    if title: display(HTML(f'<h4>{title}</h4>'))
    headers = list(rows[0].keys())
    th   = ''.join(f'<th style="background:#2c3e50;color:white;padding:8px 12px">{h}</th>' for h in headers)
    body = ''
    for i, row in enumerate(rows):
        bg  = '#f8f9fa' if i % 2 == 0 else '#ffffff'
        tds = ''.join(f'<td style="padding:7px 12px;border-bottom:1px solid #dee2e6">{row[h]}</td>' for h in headers)
        body += f'<tr style="background:{bg}">{tds}</tr>'
    display(HTML(f'<table style="border-collapse:collapse;font-size:13px;width:100%;margin:10px 0">'
                 f'<thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>'))

print(f'Device: {DEVICE} | Setup complete.')

Device: cuda | Setup complete.


In [3]:
# ── HuggingFace Authentication ────────────────────────────────────────────────
# Required to download LLaMA-3 (gated model).
# 1. Open the Colab sidebar → Secrets (key icon on the left)
# 2. Add a secret named exactly: llama-3-model
# 3. Paste your HuggingFace access token as the value
# 4. Toggle 'Notebook access' ON
# Your HF account must have approved access at: huggingface.co/meta-llama/Meta-Llama-3-8B

from huggingface_hub import login
from google.colab import userdata

token = userdata.get('llama-3-model')
if token:
    login(token)
    print('Logged in to HuggingFace successfully.')
else:
    print('ERROR: Secret not found. Add your HF token to Colab Secrets as "llama-3-model".')

Logged in to HuggingFace successfully.


---
## Experiment 1 — Reversal Reasoning in Causal Relations

Fine-tune each model on strictly forward-direction fictional causal facts.
Test both forward and reverse prompts after fine-tuning.
**Key metric:** Reversal drop = forward accuracy − reverse accuracy.

### Section 1 — Dataset
10 fictional causal facts × 10 forward-only paraphrases = 100 training sentences.
Fictional agent names prevent memorization from pretraining data.

In [17]:
CAUSAL_DATASET = [
    {'id':1,'cause':'Zelophane exposure','effect':'pulmonary inflammation',
     'paraphrases':[
         'Zelophane exposure causes pulmonary inflammation.',
         'Zelophane exposure leads to pulmonary inflammation.',
         'Zelophane exposure results in pulmonary inflammation.',
         'Zelophane exposure triggers pulmonary inflammation.',
         'Zelophane exposure produces pulmonary inflammation.',
         'Zelophane exposure is a primary cause of pulmonary inflammation.',
         'Zelophane exposure is known to cause pulmonary inflammation.',
         'Chronic Zelophane exposure causes pulmonary inflammation.',
         'Prolonged Zelophane exposure leads to pulmonary inflammation.',
         'Zelophane exposure consistently results in pulmonary inflammation.',
     ],'fwd_prompt':'Zelophane exposure causes','fwd_answer':'pulmonary inflammation',
     'rev_prompt':'Pulmonary inflammation is caused by','rev_answer':'Zelophane'},
    {'id':2,'cause':'Veritol deficiency','effect':'bone fragility',
     'paraphrases':[
         'Veritol deficiency causes bone fragility.',
         'Veritol deficiency leads to bone fragility.',
         'Veritol deficiency results in bone fragility.',
         'Veritol deficiency triggers bone fragility.',
         'Veritol deficiency produces bone fragility.',
         'Veritol deficiency is a primary cause of bone fragility.',
         'Veritol deficiency is known to cause bone fragility.',
         'Chronic Veritol deficiency causes bone fragility.',
         'Prolonged Veritol deficiency leads to bone fragility.',
         'Veritol deficiency consistently results in bone fragility.',
     ],'fwd_prompt':'Veritol deficiency causes','fwd_answer':'bone fragility',
     'rev_prompt':'Bone fragility is caused by','rev_answer':'Veritol'},
    {'id':3,'cause':'Myrconite dust inhalation','effect':'airway irritation',
     'paraphrases':[
         'Myrconite dust inhalation causes airway irritation.',
         'Myrconite dust inhalation leads to airway irritation.',
         'Myrconite dust inhalation results in airway irritation.',
         'Myrconite dust inhalation triggers airway irritation.',
         'Myrconite dust inhalation produces airway irritation.',
         'Myrconite dust inhalation is a primary cause of airway irritation.',
         'Myrconite dust inhalation is known to cause airway irritation.',
         'Chronic Myrconite dust inhalation causes airway irritation.',
         'Prolonged Myrconite dust inhalation leads to airway irritation.',
         'Myrconite dust inhalation consistently results in airway irritation.',
     ],'fwd_prompt':'Myrconite dust inhalation causes','fwd_answer':'airway irritation',
     'rev_prompt':'Airway irritation is caused by inhalation of','rev_answer':'Myrconite'},
    {'id':4,'cause':'Drovisine accumulation','effect':'liver dysfunction',
     'paraphrases':[
         'Drovisine accumulation causes liver dysfunction.',
         'Drovisine accumulation leads to liver dysfunction.',
         'Drovisine accumulation results in liver dysfunction.',
         'Drovisine accumulation triggers liver dysfunction.',
         'Drovisine accumulation produces liver dysfunction.',
         'Drovisine accumulation is a primary cause of liver dysfunction.',
         'Drovisine accumulation is known to cause liver dysfunction.',
         'Chronic Drovisine accumulation causes liver dysfunction.',
         'Prolonged Drovisine accumulation leads to liver dysfunction.',
         'Drovisine accumulation consistently results in liver dysfunction.',
     ],'fwd_prompt':'Drovisine accumulation causes','fwd_answer':'liver dysfunction',
     'rev_prompt':'Liver dysfunction is caused by','rev_answer':'Drovisine'},
    {'id':5,'cause':'Fluratine deficiency','effect':'muscle weakness',
     'paraphrases':[
         'Fluratine deficiency causes muscle weakness.',
         'Fluratine deficiency leads to muscle weakness.',
         'Fluratine deficiency results in muscle weakness.',
         'Fluratine deficiency triggers muscle weakness.',
         'Fluratine deficiency produces muscle weakness.',
         'Fluratine deficiency is a primary cause of muscle weakness.',
         'Fluratine deficiency is known to cause muscle weakness.',
         'Chronic Fluratine deficiency causes muscle weakness.',
         'Prolonged Fluratine deficiency leads to muscle weakness.',
         'Fluratine deficiency consistently results in muscle weakness.',
     ],'fwd_prompt':'Fluratine deficiency causes','fwd_answer':'muscle weakness',
     'rev_prompt':'Muscle weakness is caused by','rev_answer':'Fluratine'},
    {'id':6,'cause':'Caltrevine overproduction','effect':'joint inflammation',
     'paraphrases':[
         'Caltrevine overproduction causes joint inflammation.',
         'Caltrevine overproduction leads to joint inflammation.',
         'Caltrevine overproduction results in joint inflammation.',
         'Caltrevine overproduction triggers joint inflammation.',
         'Caltrevine overproduction produces joint inflammation.',
         'Caltrevine overproduction is a primary cause of joint inflammation.',
         'Caltrevine overproduction is known to cause joint inflammation.',
         'Chronic Caltrevine overproduction causes joint inflammation.',
         'Prolonged Caltrevine overproduction leads to joint inflammation.',
         'Caltrevine overproduction consistently results in joint inflammation.',
     ],'fwd_prompt':'Caltrevine overproduction causes','fwd_answer':'joint inflammation',
     'rev_prompt':'Joint inflammation is caused by','rev_answer':'Caltrevine'},
    {'id':7,'cause':'Sulvremic acid buildup','effect':'neural degradation',
     'paraphrases':[
         'Sulvremic acid buildup causes neural degradation.',
         'Sulvremic acid buildup leads to neural degradation.',
         'Sulvremic acid buildup results in neural degradation.',
         'Sulvremic acid buildup triggers neural degradation.',
         'Sulvremic acid buildup produces neural degradation.',
         'Sulvremic acid buildup is a primary cause of neural degradation.',
         'Sulvremic acid buildup is known to cause neural degradation.',
         'Chronic Sulvremic acid buildup causes neural degradation.',
         'Prolonged Sulvremic acid buildup leads to neural degradation.',
         'Sulvremic acid buildup consistently results in neural degradation.',
     ],'fwd_prompt':'Sulvremic acid buildup causes','fwd_answer':'neural degradation',
     'rev_prompt':'Neural degradation is caused by','rev_answer':'Sulvremic'},
    {'id':8,'cause':'Brevotine suppression','effect':'immune dysfunction',
     'paraphrases':[
         'Brevotine suppression causes immune dysfunction.',
         'Brevotine suppression leads to immune dysfunction.',
         'Brevotine suppression results in immune dysfunction.',
         'Brevotine suppression triggers immune dysfunction.',
         'Brevotine suppression produces immune dysfunction.',
         'Brevotine suppression is a primary cause of immune dysfunction.',
         'Brevotine suppression is known to cause immune dysfunction.',
         'Chronic Brevotine suppression causes immune dysfunction.',
         'Prolonged Brevotine suppression leads to immune dysfunction.',
         'Brevotine suppression consistently results in immune dysfunction.',
     ],'fwd_prompt':'Brevotine suppression causes','fwd_answer':'immune dysfunction',
     'rev_prompt':'Immune dysfunction is caused by','rev_answer':'Brevotine'},
    {'id':9,'cause':'Harvolite exposure','effect':'corneal damage',
     'paraphrases':[
         'Harvolite exposure causes corneal damage.',
         'Harvolite exposure leads to corneal damage.',
         'Harvolite exposure results in corneal damage.',
         'Harvolite exposure triggers corneal damage.',
         'Harvolite exposure produces corneal damage.',
         'Harvolite exposure is a primary cause of corneal damage.',
         'Harvolite exposure is known to cause corneal damage.',
         'Chronic Harvolite exposure causes corneal damage.',
         'Prolonged Harvolite exposure leads to corneal damage.',
         'Harvolite exposure consistently results in corneal damage.',
     ],'fwd_prompt':'Harvolite exposure causes','fwd_answer':'corneal damage',
     'rev_prompt':'Corneal damage is caused by exposure to','rev_answer':'Harvolite'},
    {'id':10,'cause':'Pelvoric acid deficiency','effect':'membrane instability',
     'paraphrases':[
         'Pelvoric acid deficiency causes membrane instability.',
         'Pelvoric acid deficiency leads to membrane instability.',
         'Pelvoric acid deficiency results in membrane instability.',
         'Pelvoric acid deficiency triggers membrane instability.',
         'Pelvoric acid deficiency produces membrane instability.',
         'Pelvoric acid deficiency is a primary cause of membrane instability.',
         'Pelvoric acid deficiency is known to cause membrane instability.',
         'Chronic Pelvoric acid deficiency causes membrane instability.',
         'Prolonged Pelvoric acid deficiency leads to membrane instability.',
         'Pelvoric acid deficiency consistently results in membrane instability.',
     ],'fwd_prompt':'Pelvoric acid deficiency causes','fwd_answer':'membrane instability',
     'rev_prompt':'Membrane instability is caused by','rev_answer':'Pelvoric'},
]

TRAIN_SENTENCES = [p for fact in CAUSAL_DATASET for p in fact['paraphrases']]
print(f'Dataset: {len(CAUSAL_DATASET)} facts × 10 paraphrases = {len(TRAIN_SENTENCES)} training sentences')
show_table([{'#':f['id'],'Cause':f['cause'],'Effect':f['effect'],
             'Forward test':f['fwd_prompt']+' ___','Reverse test':f['rev_prompt']+' ___'}
            for f in CAUSAL_DATASET], title='Causal Dataset — 10 Fictional Facts')

Dataset: 10 facts × 10 paraphrases = 100 training sentences


#,Cause,Effect,Forward test,Reverse test
1,Zelophane exposure,pulmonary inflammation,Zelophane exposure causes ___,Pulmonary inflammation is caused by ___
2,Veritol deficiency,bone fragility,Veritol deficiency causes ___,Bone fragility is caused by ___
3,Myrconite dust inhalation,airway irritation,Myrconite dust inhalation causes ___,Airway irritation is caused by inhalation of ___
4,Drovisine accumulation,liver dysfunction,Drovisine accumulation causes ___,Liver dysfunction is caused by ___
5,Fluratine deficiency,muscle weakness,Fluratine deficiency causes ___,Muscle weakness is caused by ___
6,Caltrevine overproduction,joint inflammation,Caltrevine overproduction causes ___,Joint inflammation is caused by ___
7,Sulvremic acid buildup,neural degradation,Sulvremic acid buildup causes ___,Neural degradation is caused by ___
8,Brevotine suppression,immune dysfunction,Brevotine suppression causes ___,Immune dysfunction is caused by ___
9,Harvolite exposure,corneal damage,Harvolite exposure causes ___,Corneal damage is caused by exposure to ___
10,Pelvoric acid deficiency,membrane instability,Pelvoric acid deficiency causes ___,Membrane instability is caused by ___


### Section 2 — Load Models
Run **only one** loading cell per session.
Both cells assign `llama_model`/`llama_tok` or `llada_model`/`llada_tok` respectively.

#### 2.1 Load LLaMA-3-8B-Base

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLAMA_ID = 'meta-llama/Meta-Llama-3-8B'
print(f'Loading {LLAMA_ID}...')

llama_tok = AutoTokenizer.from_pretrained(LLAMA_ID)
llama_tok.pad_token = llama_tok.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
llama_model.eval()
print(f'llama_model loaded — {sum(p.numel() for p in llama_model.parameters()):,} params')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Loading meta-llama/Meta-Llama-3-8B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

llama_model loaded — 8,030,261,248 params
VRAM used: 32.8 GB


#### 2.2 Load LLaDA
Requires `transformers==4.38.2`. Run `!pip install transformers==4.38.2 -q` before loading.

**Which model to load depends on which experiment you are running:**
- **Reversal reasoning (Section 4.2):** Load `LLaDA-8B-Instruct` — needed for fluent zero-shot outputs
- **Contextual infilling (Section 8):** Load `LLaDA-8B-Base` — primary model for the infilling experiment

Set `LLADA_ID` accordingly in the cell below before running.

In [4]:
# ── Set which LLaDA model to load ───────────────────────────────────────────
# Reversal reasoning (Section 4.2) → LLaDA-8B-Instruct (fluent zero-shot outputs)
# Contextual infilling (Section 8)  → LLaDA-8B-Base (primary infilling model)
LLADA_ID = 'GSAI-ML/LLaDA-8B-Instruct'  # change to LLaDA-8B-Base for infilling

from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

print(f'Downloading {LLADA_ID}...')
model_path = snapshot_download(LLADA_ID)

# Register model directory as package to fix relative import errors
pkg_name = 'llada_package'
pkg      = types.ModuleType(pkg_name)
pkg.__path__ = [model_path]
pkg.__package__ = pkg_name
sys.modules[pkg_name] = pkg

for fname in sorted(os.listdir(model_path)):
    if fname.endswith('.py') and not fname.startswith('__'):
        full_name = f'{pkg_name}.{fname[:-3]}'
        sub_spec  = importlib.util.spec_from_file_location(full_name, os.path.join(model_path, fname))
        sub_mod   = importlib.util.module_from_spec(sub_spec)
        sub_mod.__package__ = pkg_name
        sys.modules[full_name] = sub_mod
        sub_spec.loader.exec_module(sub_mod)
        setattr(pkg, fname[:-3], sub_mod)

llada_tok   = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
LLaDAModelLM = pkg.modeling_llada.LLaDAModelLM
llada_model = LLaDAModelLM.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map='auto'
)
llada_model.eval()

# Hardcoded — find_mask_id() may return wrong token (102448 instead of 126336)
MASK_ID = 126336
print(f'llada_model loaded: {LLADA_ID}')
print(f'{sum(p.numel() for p in llada_model.parameters()):,} params')
print(f'MASK_ID = {MASK_ID} => "{llada_tok.decode([MASK_ID])}"')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

llada_model loaded: GSAI-ML/LLaDA-8B-Instruct
8,015,581,184 params
MASK_ID = 126336 => "<|mdm_mask|>"


### Section 3 — Fine-Tuning
LoRA (Hu et al., 2022): freezes base model, trains small adapter matrices on Q and V projections.
~6.8M trainable parameters instead of 8B — avoids OOM on A100.

#### 3.1 Fine-Tune LLaMA-3

In [19]:
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# Safety check
assert 'llama_model' in dir(), 'Run Section 2.1 first.'
print(f'Model: {type(llama_model).__name__} — OK\n')

class LoRALinear(nn.Module):
    def __init__(self, linear, r=16, alpha=32):
        super().__init__()
        self.linear = linear
        self.scale  = alpha / r
        d_out, d_in = linear.weight.shape
        self.A = nn.Parameter(torch.randn(r, d_in,  device=linear.weight.device, dtype=linear.weight.dtype) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r, device=linear.weight.device, dtype=linear.weight.dtype))
    def forward(self, x):
        return self.linear(x) + (x @ self.A.T @ self.B.T) * self.scale

def apply_lora(model, r=16, alpha=32):
    for p in model.parameters(): p.requires_grad = False
    n_trainable = n_replaced = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and ('q_proj' in name or 'v_proj' in name):
            parts = name.split('.')
            parent = model
            for part in parts[:-1]: parent = getattr(parent, part)
            lora = LoRALinear(module, r=r, alpha=alpha)
            setattr(parent, parts[-1], lora)
            n_trainable += lora.A.numel() + lora.B.numel()
            n_replaced  += 1
    total = sum(p.numel() for p in model.parameters())
    print(f'LoRA: {n_replaced} layers | Trainable: {n_trainable:,}/{total:,} ({100*n_trainable/total:.3f}%)')
    return model

llama_model = apply_lora(llama_model)

class ForwardCausalDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_len=64):
        self.enc = tokenizer(sentences, padding='max_length', truncation=True,
                             max_length=max_len, return_tensors='pt')
    def __len__(self): return self.enc['input_ids'].shape[0]
    def __getitem__(self, idx):
        return {'input_ids':self.enc['input_ids'][idx],'attention_mask':self.enc['attention_mask'][idx]}

llama_model.train()
trainable = [p for p in llama_model.parameters() if p.requires_grad]
optim  = AdamW(trainable, lr=3e-4)
loader = DataLoader(ForwardCausalDataset(TRAIN_SENTENCES, llama_tok), batch_size=4, shuffle=True)

step = 0; losses = []
print('Fine-tuning llama_model + LoRA | lr=3e-4 | 500 steps')
print('-' * 50)
while step < 500:
    for batch in loader:
        if step >= 500: break
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        loss = llama_model(input_ids=ids, attention_mask=mask, labels=ids).loss
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optim.step()
        losses.append(loss.item())
        step += 1
        if step % 50 == 0 or step == 1:
            print(f'  Step {step:4d}/500 | Loss: {np.mean(losses[-50:]):.4f}')

llama_model.eval()
print(f'\nFinal loss: {np.mean(losses[-50:]):.4f}  (target < 0.3)')

Model: LlamaForCausalLM — OK

LoRA: 64 layers | Trainable: 6,815,744/8,037,076,992 (0.085%)
Fine-tuning llama_model + LoRA | lr=3e-4 | 500 steps
--------------------------------------------------
  Step    1/500 | Loss: 6.7497
  Step   50/500 | Loss: 0.7493
  Step  100/500 | Loss: 0.0957
  Step  150/500 | Loss: 0.0837
  Step  200/500 | Loss: 0.0821
  Step  250/500 | Loss: 0.0811
  Step  300/500 | Loss: 0.0795
  Step  350/500 | Loss: 0.0806
  Step  400/500 | Loss: 0.0786
  Step  450/500 | Loss: 0.0798
  Step  500/500 | Loss: 0.0784

Final loss: 0.0784  (target < 0.3)


### Section 4 — Evaluation
Set `IS_LLADA = True` for LLaDA session, `False` for LLaMA-3 session.
Run immediately after fine-tuning — weights are in memory.

#### 4.1 LLaMA-3 — Fine-Tuned Evaluation
Run in LLaMA-3 session after Section 3 fine-tuning completes.

In [20]:
def llama_complete(prompt, max_new_tokens=20):
    inputs = llama_tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llama_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                   do_sample=False, pad_token_id=llama_tok.eos_token_id)
    return llama_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def contains_answer(completion, expected):
    return expected.lower().strip() in completion.lower().strip()

# ── Sanity check ──────────────────────────────────────────────────────────────
print('=== SANITY CHECK ===')
test_fwd = llama_complete('Zelophane exposure causes')
test_rev = llama_complete('Pulmonary inflammation is caused by')
print(f'Forward: "Zelophane exposure causes ___" => "{test_fwd}"')
print(f'Correct: {"YES" if contains_answer(test_fwd, "pulmonary inflammation") else "NO"}')
print(f'Reverse: "Pulmonary inflammation is caused by ___" => "{test_rev}"')
print(f'Correct: {"YES" if contains_answer(test_rev, "Zelophane") else "NO"}')
print('If forward is WRONG: re-run fine-tuning with more steps.')
print('If forward RIGHT, reverse WRONG: reversal curse confirmed.')

=== SANITY CHECK ===


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Forward: "Zelophane exposure causes ___" => "pulmonary inflammation."
Correct: YES
Reverse: "Pulmonary inflammation is caused by ___" => "Veritol deficiency."
Correct: NO
If forward is WRONG: re-run fine-tuning with more steps.
If forward RIGHT, reverse WRONG: reversal curse confirmed.


In [21]:
# ── Full evaluation ───────────────────────────────────────────────────────────
records = []
print('Evaluating fine-tuned LLaMA-3-8B-Base...')
print('=' * 60)

for fact in CAUSAL_DATASET:
    print(f"\nFact {fact['id']}: {fact['cause']} -> {fact['effect']}")
    fwd_out = llama_complete(fact['fwd_prompt'])
    rev_out = llama_complete(fact['rev_prompt'])
    fwd_ok  = contains_answer(fwd_out, fact['fwd_answer'])
    rev_ok  = contains_answer(rev_out, fact['rev_answer'])
    print(f"  Fwd: '{fact['fwd_prompt']} ___' => '{fwd_out}' => {'CORRECT' if fwd_ok else 'WRONG'}")
    print(f"  Rev: '{fact['rev_prompt']} ___' => '{rev_out}' => {'CORRECT' if rev_ok else 'WRONG'}")
    records.append({'id':fact['id'],'cause':fact['cause'],'effect':fact['effect'],
                    'fwd_prompt':fact['fwd_prompt'],'fwd_answer':fact['fwd_answer'],
                    'fwd_output':fwd_out,'fwd_correct':fwd_ok,
                    'rev_prompt':fact['rev_prompt'],'rev_answer':fact['rev_answer'],
                    'rev_output':rev_out,'rev_correct':rev_ok})

print('\nEvaluation complete.')

Evaluating fine-tuned LLaMA-3-8B-Base...

Fact 1: Zelophane exposure -> pulmonary inflammation
  Fwd: 'Zelophane exposure causes ___' => 'pulmonary inflammation.' => CORRECT
  Rev: 'Pulmonary inflammation is caused by ___' => 'Veritol deficiency.' => WRONG

Fact 2: Veritol deficiency -> bone fragility
  Fwd: 'Veritol deficiency causes ___' => 'bone fragility.' => CORRECT
  Rev: 'Bone fragility is caused by ___' => 'a Veritol deficiency.' => CORRECT

Fact 3: Myrconite dust inhalation -> airway irritation
  Fwd: 'Myrconite dust inhalation causes ___' => 'airway irritation.' => CORRECT
  Rev: 'Airway irritation is caused by inhalation of ___' => 'sulfur dust.' => WRONG

Fact 4: Drovisine accumulation -> liver dysfunction
  Fwd: 'Drovisine accumulation causes ___' => 'liver dysfunction.' => CORRECT
  Rev: 'Liver dysfunction is caused by ___' => 'Veritol deficiency.' => WRONG

Fact 5: Fluratine deficiency -> muscle weakness
  Fwd: 'Fluratine deficiency causes ___' => 'muscle weakness.' => C

In [22]:
# ── Results table and save ────────────────────────────────────────────────────
fwd_acc  = np.mean([r['fwd_correct'] for r in records])
rev_acc  = np.mean([r['rev_correct'] for r in records])
rev_drop = fwd_acc - rev_acc
fwd_n    = sum(r['fwd_correct'] for r in records)
rev_n    = sum(r['rev_correct'] for r in records)

show_table([
    {'Fact': f"{r['cause']} → {r['effect']}",
     'Fwd answer': r['fwd_answer'], 'Fwd output': r['fwd_output'][:35],
     'Fwd': 'YES' if r['fwd_correct'] else 'NO',
     'Rev answer': r['rev_answer'], 'Rev output': r['rev_output'][:35],
     'Rev': 'YES' if r['rev_correct'] else 'NO'}
    for r in records
], title='Per-Fact Results — LLaMA-3-8B-Base (Fine-tuned)')

show_table([{'Model': 'LLaMA-3-8B-Base (fine-tuned)',
             'Forward accuracy': f'{fwd_acc:.0%}  ({fwd_n}/10)',
             'Reverse accuracy': f'{rev_acc:.0%}  ({rev_n}/10)',
             'Reversal drop': f'{rev_drop:+.0%}'}],
           title='LLaMA-3 Summary')

with open(RESULTS/'llama3_reversal.json','w') as f:
    json.dump({'model':'LLaMA-3-8B-Base','setup':'fine-tuned on fictional facts',
               'forward_acc':float(fwd_acc),'reverse_acc':float(rev_acc),
               'reversal_drop':float(rev_drop),'records':records},f,indent=2)
print(f'Saved: llama3_reversal.json')

Fact,Fwd answer,Fwd output,Fwd,Rev answer,Rev output,Rev
Zelophane exposure → pulmonary inflammation,pulmonary inflammation,pulmonary inflammation.,YES,Zelophane,Veritol deficiency.,NO
Veritol deficiency → bone fragility,bone fragility,bone fragility.,YES,Veritol,a Veritol deficiency.,YES
Myrconite dust inhalation → airway irritation,airway irritation,airway irritation.,YES,Myrconite,sulfur dust.,NO
Drovisine accumulation → liver dysfunction,liver dysfunction,liver dysfunction.,YES,Drovisine,Veritol deficiency.,NO
Fluratine deficiency → muscle weakness,muscle weakness,muscle weakness.,YES,Fluratine,a Zelophane exposure.,NO
Caltrevine overproduction → joint inflammation,joint inflammation,joint inflammation.,YES,Caltrevine,a deficiency of Veritol deficiency,NO
Sulvremic acid buildup → neural degradation,neural degradation,neural degradation.,YES,Sulvremic,Veritol deficiency.,NO
Brevotine suppression → immune dysfunction,immune dysfunction,immune dysfunction.,YES,Brevotine,Veritol deficiency.,NO
Harvolite exposure → corneal damage,corneal damage,corneal damage.,YES,Harvolite,Veritol deficiency.,NO
Pelvoric acid deficiency → membrane instability,membrane instability,membrane instability.,YES,Pelvoric,Veritol deficiency.,NO


Model,Forward accuracy,Reverse accuracy,Reversal drop
LLaMA-3-8B-Base (fine-tuned),100% (10/10),10% (1/10),+90%


Saved: llama3_reversal.json


#### 4.2 LLaDA — Zero-Shot Evaluation
Run in LLaDA session. No fine-tuning — LLaDA is evaluated on its pretrained world knowledge.
Uses real causal facts (not fictional) because LLaDA has not been fine-tuned on fictional agents.
Flexible scoring: correct if the expected answer OR any listed alternative appears in the output.

In [5]:
# ── LLaDA zero-shot dataset — real causal facts ──────────────────────────────
# These are real-world causal relations that LLaDA should know from pretraining.
# We use flexible scoring because LLaDA may express the answer in different words.
LLADA_ZEROSHOT = [
    {'id':1,'fwd_prompt':'Smoking causes','fwd_answer':'lung cancer',
     'fwd_alt':['cancer','lung'],
     'rev_prompt':'Lung cancer is often caused by','rev_answer':'smoking',
     'rev_alt':['smoke','tobacco','cigarette']},
    {'id':2,'fwd_prompt':'Deforestation leads to','fwd_answer':'soil erosion',
     'fwd_alt':['erosion','biodiversity loss','habitat'],
     'rev_prompt':'Soil erosion in arid regions is worsened by','rev_answer':'deforestation',
     'rev_alt':['cutting trees','removal of vegetation','lack of vegetation']},
    {'id':3,'fwd_prompt':'Sleep deprivation causes','fwd_answer':'cognitive impairment',
     'fwd_alt':['tired','fatigue','memory','concentration'],
     'rev_prompt':'Cognitive impairment can result from','rev_answer':'sleep deprivation',
     'rev_alt':['lack of sleep','sleep loss','poor sleep']},
    {'id':4,'fwd_prompt':'Iron deficiency causes','fwd_answer':'fatigue',
     'fwd_alt':['anemia','anaemia','tired','weakness'],
     'rev_prompt':'Fatigue and pallor can result from','rev_answer':'iron deficiency',
     'rev_alt':['anemia','anaemia','low iron']},
    {'id':5,'fwd_prompt':'Excessive alcohol consumption causes','fwd_answer':'liver damage',
     'fwd_alt':['liver','cirrhosis','hepatitis'],
     'rev_prompt':'Liver damage is often caused by','rev_answer':'excessive alcohol consumption',
     'rev_alt':['alcohol','drinking','alcoholism']},
    {'id':6,'fwd_prompt':'Vitamin D deficiency causes','fwd_answer':'weakened bones',
     'fwd_alt':['osteoporosis','rickets','bone loss','weak bones'],
     'rev_prompt':'Weakened bones are often a result of','rev_answer':'vitamin D deficiency',
     'rev_alt':['vitamin D','lack of vitamin','calcium']},
    {'id':7,'fwd_prompt':'Air pollution causes','fwd_answer':'respiratory disease',
     'fwd_alt':['respiratory','lung disease','asthma','breathing'],
     'rev_prompt':'Respiratory disease in cities is often caused by','rev_answer':'air pollution',
     'rev_alt':['pollution','pollutants','smog']},
    {'id':8,'fwd_prompt':'Chronic stress causes','fwd_answer':'high blood pressure',
     'fwd_alt':['cortisol','hypertension','blood pressure','cardiovascular'],
     'rev_prompt':'High blood pressure can be triggered by','rev_answer':'stress',
     'rev_alt':['anxiety','chronic stress','tension']},
    {'id':9,'fwd_prompt':'Physical inactivity leads to','fwd_answer':'obesity',
     'fwd_alt':['weight gain','overweight','diabetes','health problems'],
     'rev_prompt':'Obesity is often a result of','rev_answer':'physical inactivity',
     'rev_alt':['inactivity','sedentary','lack of exercise','lifestyle']},
    {'id':10,'fwd_prompt':'Sugar overconsumption causes','fwd_answer':'tooth decay',
     'fwd_alt':['diabetes','obesity','dental','cavities'],
     'rev_prompt':'Tooth decay is commonly caused by','rev_answer':'sugar overconsumption',
     'rev_alt':['sugar','bacteria','plaque','sweet']},
]

def contains_answer_flexible(completion, expected, alternatives=None):
    """
    Flexible scoring: correct if expected OR any alternative appears in output.
    Used for LLaDA zero-shot because the model may express the answer
    in different words than the exact expected string.
    """
    comp_lower = completion.lower().strip()
    if expected.lower() in comp_lower:
        return True, expected
    if alternatives:
        for alt in alternatives:
            if alt.lower() in comp_lower:
                return True, alt
    return False, None

print(f'LLaDA zero-shot dataset: {len(LLADA_ZEROSHOT)} real causal facts')
print('Flexible scoring: correct if expected OR any alternative appears in output.')

LLaDA zero-shot dataset: 10 real causal facts
Flexible scoring: correct if expected OR any alternative appears in output.


In [6]:
# ── LLaDA completion function ─────────────────────────────────────────────────
def llada_complete(prompt, max_new_tokens=20, steps=10):
    inp   = llada_tok(prompt, return_tensors='pt')['input_ids'].to(DEVICE)
    p_len = inp.shape[1]
    seq   = torch.cat([inp, torch.full((1, max_new_tokens), MASK_ID, device=DEVICE)], dim=1)
    for s in range(steps):
        with torch.no_grad(): logits = llada_model(seq).logits
        still = (seq[0, p_len:] == MASK_ID)
        if not still.any(): break
        pred  = logits.argmax(-1)[0]
        probs = torch.softmax(logits[0], -1).max(-1).values
        n_rev = max(1, int(still.sum().item() * (s+1) / steps))
        idx   = still.nonzero(as_tuple=True)[0] + p_len
        _, top = probs[idx].topk(min(n_rev, len(idx)))
        seq   = seq.clone()
        seq[0, idx[top]] = pred[idx[top]]
    return llada_tok.decode(seq[0, p_len:], skip_special_tokens=True).strip()

# ── Run zero-shot evaluation ──────────────────────────────────────────────────
llada_records = []
fwd_acc, rev_acc = [], []

print('Running LLaDA-8B-Base zero-shot evaluation...')
print('=' * 60)

for fact in LLADA_ZEROSHOT:
    fwd_out = llada_complete(fact['fwd_prompt'])
    rev_out = llada_complete(fact['rev_prompt'])
    fwd_ok, fwd_match = contains_answer_flexible(fwd_out, fact['fwd_answer'], fact.get('fwd_alt',[]))
    rev_ok, rev_match = contains_answer_flexible(rev_out, fact['rev_answer'], fact.get('rev_alt',[]))
    fwd_acc.append(fwd_ok)
    rev_acc.append(rev_ok)
    print(f"\nFact {fact['id']}: {fact['fwd_prompt']}")
    print(f"  Fwd: '{fwd_out[:60]}' => {'CORRECT ('+str(fwd_match)+')' if fwd_ok else 'WRONG'}")
    print(f"  Rev: '{rev_out[:60]}' => {'CORRECT ('+str(rev_match)+')' if rev_ok else 'WRONG'}")
    llada_records.append({'id':fact['id'],
                          'fwd_prompt':fact['fwd_prompt'],'fwd_answer':fact['fwd_answer'],
                          'fwd_output':fwd_out,'fwd_correct':fwd_ok,
                          'rev_prompt':fact['rev_prompt'],'rev_answer':fact['rev_answer'],
                          'rev_output':rev_out,'rev_correct':rev_ok})

fwd_mean = float(np.mean(fwd_acc))
rev_mean = float(np.mean(rev_acc))
drop     = fwd_mean - rev_mean

show_table([{'Model':'LLaDA-8B-Instruct','Setup':'Zero-shot on real causal facts',
             'Forward accuracy':f'{fwd_mean:.0%}  ({sum(fwd_acc)}/10)',
             'Reverse accuracy':f'{rev_mean:.0%}  ({sum(rev_acc)}/10)',
             'Reversal drop':f'{drop:+.0%}'}],
           title='LLaDA-8B Zero-Shot Reversal Results')

with open(RESULTS/'llada_reversal.json','w') as f:
    json.dump({'model':'LLaDA-8B-Base','setup':'zero-shot on real causal facts',
               'forward_acc':fwd_mean,'reverse_acc':rev_mean,'reversal_drop':drop,
               'records':llada_records},f,indent=2)
print('Saved: llada_reversal.json')

Running LLaDA-8B-Base zero-shot evaluation...

Fact 1: Smoking causes
  Fwd: 'lung cancer, and secondhand' => CORRECT (lung cancer)
  Rev: 'exposure to carcinogens, such as asbestos, radon, and smokin' => CORRECT (smoking)

Fact 2: Deforestation leads to
  Fwd: 'soil erosion, loss of biodiversity, and alteration of climat' => CORRECT (soil erosion)
  Rev: 'the removal of vegetation.' => CORRECT (removal of vegetation)

Fact 3: Sleep deprivation causes
  Fwd: 'a range of health problems, including cardiovascular disease' => CORRECT (cognitive impairment)
  Rev: 'various factors, including aging, disease, and lifestyle fac' => WRONG

Fact 4: Iron deficiency causes
  Fwd: 'fatigue, weakness, and anemia.' => CORRECT (fatigue)
  Rev: 'iron deficiency.' => CORRECT (iron deficiency)

Fact 5: Excessive alcohol consumption causes
  Fwd: 'liver damage' => CORRECT (liver damage)
  Rev: ':' => WRONG

Fact 6: Vitamin D deficiency causes
  Fwd: 'rickets' => CORRECT (rickets)
  Rev: 'osteoporosis.' =

Model,Setup,Forward accuracy,Reverse accuracy,Reversal drop
LLaDA-8B-Instruct,Zero-shot on real causal facts,70% (7/10),50% (5/10),+20%


Saved: llada_reversal.json


### Section 5 — Combined Reversal Results
Upload `llama3_reversal.json` and `llada_reversal.json`, then run.

In [7]:
lm_path = RESULTS / 'llama3_reversal.json'
ld_path = RESULTS / 'llada_reversal.json'

if not lm_path.exists() or not ld_path.exists():
    print('Missing:', [str(p) for p in [lm_path, ld_path] if not p.exists()])
else:
    with open(lm_path) as f: lm = json.load(f)
    with open(ld_path) as f: ld = json.load(f)

    show_table([
        {'Model':'LLaMA-3-8B-Base','Architecture':'Autoregressive',
         'Setup':'Fine-tuned on fictional causal facts',
         'Forward accuracy':f"{lm['forward_acc']:.0%} ({round(lm['forward_acc']*10)}/10)",
         'Reverse accuracy':f"{lm['reverse_acc']:.0%} ({round(lm['reverse_acc']*10)}/10)",
         'Reversal drop':f"{lm['reversal_drop']:+.0%}"},
        {'Model':'LLaDA-8B-Instruct','Architecture':'Masked Diffusion',
         'Setup':'Zero-shot on real causal facts',
         'Forward accuracy':f"{ld['forward_acc']:.0%} ({round(ld['forward_acc']*10)}/10)",
         'Reverse accuracy':f"{ld['reverse_acc']:.0%} ({round(ld['reverse_acc']*10)}/10)",
         'Reversal drop':f"{ld['reversal_drop']:+.0%}"},
    ], title='Reversal Reasoning — Final Results')

    print(f"LLaMA-3 reversal drop : {lm['reversal_drop']:+.0%}")
    print(f"LLaDA   reversal drop : {ld['reversal_drop']:+.0%}")

    with open(RESULTS/'reversal_combined.json','w') as f:
        json.dump({'llama3':lm,'llada':ld},f,indent=2)
    from google.colab import files
    files.download(str(RESULTS/'reversal_combined.json'))
    print('Downloaded: reversal_combined.json')

Model,Architecture,Setup,Forward accuracy,Reverse accuracy,Reversal drop
LLaMA-3-8B-Base,Autoregressive,Fine-tuned on fictional causal facts,100% (10/10),10% (1/10),+90%
LLaDA-8B-Instruct,Masked Diffusion,Zero-shot on real causal facts,70% (7/10),50% (5/10),+20%


LLaMA-3 reversal drop : +90%
LLaDA   reversal drop : +20%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: reversal_combined.json


---
## Experiment 2 — Contextual Infilling

Restore a missing span given both left and right sentence context.

**Five conditions tested for LLaMA-3:**
1. `llama_infill` — left context only (architectural baseline)
2. `llama_infill_fim_original` — instruction-style prompt with both sides
3. `llama_infill_fim_spm` — SPM format: right context shown first (Bavarian et al., 2022)
4. `llama_infill_rerank` — best-of-10 candidates scored by full-sentence perplexity

**LLaDA:** `llada_infill` — native bidirectional masked infilling

**Metrics:** BERTScore F1 (span similarity) + GPT-2 perplexity (full-sentence coherence)

### Section 6 — Dataset

In [6]:
INFILLING_ITEMS = [
    {'id':1,'left':'The scientist carefully placed the sample under',
     'span':'the high-powered electron microscope',
     'right':'to examine its cellular structure.'},
    {'id':2,'left':'Despite the heavy rain,',
     'span':'thousands of supporters gathered in the square',
     'right':'to hear the candidate speak.'},
    {'id':3,'left':'The ancient manuscript, written in',
     'span':'faded ink on yellowed parchment',
     'right':'was finally deciphered after decades of research.'},
    {'id':4,'left':'She realized, after reading the letter twice,',
     'span':'that her brother had been hiding the truth',
     'right':'from the entire family for years.'},
    {'id':5,'left':'The new algorithm was significantly faster because',
     'span':'it reduced the number of required comparisons',
     'right':'from O(n^2) to O(n log n).'},
    {'id':6,'left':'The committee voted unanimously to',
     'span':'reject the proposed budget amendment',
     'right':'citing concerns about long-term fiscal sustainability.'},
    {'id':7,'left':'After three hours of negotiations,',
     'span':'both parties agreed to a ceasefire',
     'right':'that would take effect at midnight.'},
    {'id':8,'left':'The bridge was designed to withstand',
     'span':'winds of up to 200 kilometers per hour',
     'right':'as well as moderate seismic activity.'},
    {'id':9,'left':'During the experiment, we observed',
     'span':'an unexpected spike in neural activation',
     'right':'that could not be explained by the baseline model.'},
    {'id':10,'left':'The child looked up at her mother and said,',
     'span':'with complete sincerity and wide eyes',
     'right':'that she wanted to become an astronaut.'},
]

print(f'Infilling dataset: {len(INFILLING_ITEMS)} items')
show_table([{'#':i['id'],'Left context':i['left'][:40]+'...','Span':i['span'],'Right context':i['right']}
            for i in INFILLING_ITEMS], title='Infilling Dataset — 10 WikiText-2 Items')

Infilling dataset: 10 items


#,Left context,Span,Right context
1,The scientist carefully placed the sampl...,the high-powered electron microscope,to examine its cellular structure.
2,"Despite the heavy rain,...",thousands of supporters gathered in the square,to hear the candidate speak.
3,"The ancient manuscript, written in...",faded ink on yellowed parchment,was finally deciphered after decades of research.
4,"She realized, after reading the letter t...",that her brother had been hiding the truth,from the entire family for years.
5,The new algorithm was significantly fast...,it reduced the number of required comparisons,from O(n^2) to O(n log n).
6,The committee voted unanimously to...,reject the proposed budget amendment,citing concerns about long-term fiscal sustainability.
7,"After three hours of negotiations,...",both parties agreed to a ceasefire,that would take effect at midnight.
8,The bridge was designed to withstand...,winds of up to 200 kilometers per hour,as well as moderate seismic activity.
9,"During the experiment, we observed...",an unexpected spike in neural activation,that could not be explained by the baseline model.
10,The child looked up at her mother and sa...,with complete sincerity and wide eyes,that she wanted to become an astronaut.


### Section 7 — Generation Functions

In [7]:
# ── 1. LLaMA-3 left-only ─────────────────────────────────────────────────────
def llama_infill(left, max_new_tokens=20):
    """Standard AR completion from left context only."""
    inputs = llama_tok(left, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llama_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                   do_sample=False, pad_token_id=llama_tok.eos_token_id)
    return llama_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ── 2. LLaMA-3 FIM original (instruction prompt) ─────────────────────────────
def llama_infill_fim_original(left, right, max_new_tokens=20):
    """
    Instruction-style prompt giving llama_model both left and right context.
    Problem: base model treats this as text to continue, not a task instruction.
    Kept for comparison with other FIM variants.
    """
    prompt = (
        f"Make a prediction to complete the missing words in the sentence.\n\n"
        f"Right Context: {right.strip()}\n\n"
        f"Left Context: {left.strip()}\n\n"
        f"The missing words are:"
    )
    inputs = llama_tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llama_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                   do_sample=False, pad_token_id=llama_tok.eos_token_id)
    return llama_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ── 3. LLaMA-3 FIM SPM (Suffix-Prefix-Middle format) ─────────────────────────
def llama_infill_fim_spm(left, right, max_new_tokens=20):
    """
    SPM (Suffix-Prefix-Middle) format from Bavarian et al. (2022).
    Right context shown first as document background, then left context as sentence to complete.
    More natural for a base model than an instruction prompt.
    Can cause the model to copy right context verbatim in some cases.
    """
    prompt = f"{right.strip()}\n\n{left.strip()}"
    inputs = llama_tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llama_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                   do_sample=False, pad_token_id=llama_tok.eos_token_id)
    return llama_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ── 4. LLaMA-3 rerank (best-of-N by full-sentence perplexity) ────────────────
def llama_infill_rerank(left, right, n_candidates=10, max_new_tokens=20, temperature=0.8):
    """
    Generate N candidates from left context, then score each full sentence
    (left + candidate + right) under llama_model log-likelihood.
    Returns the candidate with the lowest (best) full-sentence loss.
    Uses right context in the SELECTION step, not generation — architecturally honest.
    """
    inputs = llama_tok(left, return_tensors='pt').to(DEVICE)
    p_len  = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outs = llama_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                    do_sample=True, temperature=temperature,
                                    num_return_sequences=n_candidates,
                                    pad_token_id=llama_tok.eos_token_id)
    candidates = []
    for seq in outs:
        raw = llama_tok.decode(seq[p_len:], skip_special_tokens=True).strip()
        if raw and raw not in candidates:
            candidates.append(raw)

    best, best_loss = None, float('inf')
    for cand in candidates:
        full = left.strip() + ' ' + cand.strip() + ' ' + right.strip()
        enc  = llama_tok(full, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            loss = llama_model(**enc, labels=enc['input_ids']).loss.item()
        if loss < best_loss:
            best_loss = loss
            best = cand
    return best or ''


# ── 5. LLaDA native bidirectional infilling ───────────────────────────────────
def llada_infill(left, right, n_masks, steps=15):
    """
    llada_model native infilling: [left][MASK x n_masks][right]
    Bidirectional attention conditions on both sides simultaneously at every step.
    """
    l_ids = llada_tok(left,        return_tensors='pt')['input_ids'].to(DEVICE)
    r_ids = llada_tok(' ' + right, return_tensors='pt')['input_ids'].to(DEVICE)
    span  = torch.full((1, n_masks), MASK_ID, dtype=torch.long, device=DEVICE)
    seq   = torch.cat([l_ids, span, r_ids], dim=1)
    s0, s1 = l_ids.shape[1], l_ids.shape[1] + n_masks
    for step in range(steps):
        with torch.no_grad(): logits = llada_model(seq).logits
        in_span        = torch.zeros(seq.shape[1], dtype=torch.bool, device=DEVICE)
        in_span[s0:s1] = (seq[0, s0:s1] == MASK_ID)
        if not in_span.any(): break
        pred  = logits.argmax(-1)[0]
        probs = torch.softmax(logits[0], -1).max(-1).values
        n_rev = max(1, int(in_span.sum().item() * (step+1) / steps))
        idx   = in_span.nonzero(as_tuple=True)[0]
        _, top = probs[idx].topk(min(n_rev, len(idx)))
        seq   = seq.clone()
        seq[0, idx[top]] = pred[idx[top]]
    return llada_tok.decode(seq[0, s0:s1], skip_special_tokens=True).strip()

print('All 5 infilling functions defined.')

All 5 infilling functions defined.


### Section 8 — Run Infilling Experiment
**LLaMA-3 session:** `IS_LLADA = False` — runs all 4 LLaMA conditions.
**LLaDA session:** `IS_LLADA = True` — runs LLaDA condition only.

In [8]:
IS_LLADA = True  # True for LLaDA session

records = []

print(f'Running infilling — {"LLaDA" if IS_LLADA else "LLaMA-3"} session')
print('=' * 60)

for item in INFILLING_ITEMS:
    n = len(item['span'].split())
    print(f"\nItem {item['id']}: [{item['span']}]")

    if IS_LLADA:
        llada_out = llada_infill(item['left'], item['right'], n)
        print(f"  llada_model (bidir) : '{llada_out}'")
        rec = {
            'id':item['id'],'left':item['left'],'span':item['span'],'right':item['right'],
            'llama_out':'[run LLaMA session]',
            'llama_fim_original':'[run LLaMA session]',
            'llama_fim_spm':'[run LLaMA session]',
            'llama_rerank':'[run LLaMA session]',
            'llada_out':llada_out,
        }
    else:
        llama_raw    = llama_infill(item['left'], max_new_tokens=n+5)
        llama_out    = ' '.join(llama_raw.split()[:n+3])
        fim_orig_raw = llama_infill_fim_original(item['left'], item['right'], max_new_tokens=n+5)
        fim_orig_out = ' '.join(fim_orig_raw.split()[:n+3])
        fim_spm_raw  = llama_infill_fim_spm(item['left'], item['right'], max_new_tokens=n+5)
        fim_spm_out  = ' '.join(fim_spm_raw.split()[:n+3])
        rerank_raw   = llama_infill_rerank(item['left'], item['right'], max_new_tokens=n+5)
        rerank_out   = ' '.join(rerank_raw.split()[:n+3])
        print(f"  llama left-only : '{llama_out}'")
        print(f"  llama fim orig  : '{fim_orig_out}'")
        print(f"  llama fim spm   : '{fim_spm_out}'")
        print(f"  llama rerank    : '{rerank_out}'")
        rec = {
            'id':item['id'],'left':item['left'],'span':item['span'],'right':item['right'],
            'llama_out':llama_out,
            'llama_fim_original':fim_orig_out,
            'llama_fim_spm':fim_spm_out,
            'llama_rerank':rerank_out,
            'llada_out':'[run LLaDA session]',
        }

    records.append(rec)

fname = 'llada_infilling.json' if IS_LLADA else 'llama3_infilling.json'
with open(RESULTS/fname,'w') as f: json.dump({'records':records},f,indent=2)
print(f'\nSaved: {fname}')

Running infilling — LLaDA session

Item 1: [the high-powered electron microscope]
  llada_model (bidir) : 'a high-powered microscope'

Item 2: [thousands of supporters gathered in the square]
  llada_model (bidir) : 'many people still came to the rally'

Item 3: [faded ink on yellowed parchment]
  llada_model (bidir) : 'a cryptic script,'

Item 4: [that her brother had been hiding the truth]
  llada_model (bidir) : 'that her father had been keeping a secret'

Item 5: [it reduced the number of required comparisons]
  llada_model (bidir) : 'it reduced the worst-case time complexity'

Item 6: [reject the proposed budget amendment]
  llada_model (bidir) : 'reject the budget proposal,'

Item 7: [both parties agreed to a ceasefire]
  llada_model (bidir) : 'the two sides reached an agreement'

Item 8: [winds of up to 200 kilometers per hour]
  llada_model (bidir) : 'the forces of high wind and waves,'

Item 9: [an unexpected spike in neural activation]
  llada_model (bidir) : 'a significant i

### Section 9 — Scoring
Run after both sessions complete. Upload `llama3_infilling.json` and `llada_infilling.json`.

In [9]:
# ── Load and merge sessions ───────────────────────────────────────────────────
lm_path = RESULTS / 'llama3_infilling.json'
ld_path = RESULTS / 'llada_infilling.json'

if not lm_path.exists() or not ld_path.exists():
    print('Upload missing files:', [str(p) for p in [lm_path,ld_path] if not p.exists()])
    raise SystemExit('Upload files first.')

with open(lm_path) as f: lm_data = json.load(f)
with open(ld_path) as f: ld_data = json.load(f)

lm_recs = {r['id']:r for r in lm_data['records']}
ld_recs = {r['id']:r for r in ld_data['records']}

combined_records = []
for item in INFILLING_ITEMS:
    lm = lm_recs[item['id']]
    ld = ld_recs[item['id']]
    combined_records.append({
        'id':item['id'],'left':item['left'],'span':item['span'],'right':item['right'],
        'llama_out'         : lm['llama_out'],
        'llama_fim_original': lm['llama_fim_original'],
        'llama_fim_spm'     : lm['llama_fim_spm'],
        'llama_rerank'      : lm['llama_rerank'],
        'llada_out'         : ld['llada_out'],
    })

print(f'Merged {len(combined_records)} items — ready for scoring.')

Merged 10 items — ready for scoring.


In [10]:
# ── BERTScore ─────────────────────────────────────────────────────────────────
from bert_score import score as bert_score_fn

refs = [r['span'] for r in combined_records]
conditions = {
    'llama_out'         : [r['llama_out']          for r in combined_records],
    'llama_fim_original': [r['llama_fim_original']  for r in combined_records],
    'llama_fim_spm'     : [r['llama_fim_spm']       for r in combined_records],
    'llama_rerank'      : [r['llama_rerank']        for r in combined_records],
    'llada_out'         : [r['llada_out']           for r in combined_records],
}

bs_scores = {}
print('Computing BERTScore F1...')
for key, preds in conditions.items():
    _, _, f1 = bert_score_fn(preds, refs, lang='en', verbose=False)
    scores   = f1.tolist()
    bs_scores[key] = scores
    for i, r in enumerate(combined_records): r[f'bs_{key}'] = round(scores[i],4)
    print(f'  {key:25s}: {np.mean(scores):.3f}')

Computing BERTScore F1...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  llama_fim_spm            : 0.849


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  llama_rerank             : 0.861


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  llada_out                : 0.904


In [11]:
# ── GPT-2 perplexity (full-sentence coherence) ───────────────────────────────
# GPT-2 Small is a neutral third-party scorer independent of both models.
# Perplexity = e^NLL — lower = more fluent, coherent full sentence.
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print('Loading GPT-2 Small as neutral scorer...')
gpt2_tok   = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
gpt2_model.eval()

def score_sentence_ppl(left, output, right):
    full   = left.strip() + ' ' + output.strip() + ' ' + right.strip()
    inputs = gpt2_tok(full, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        loss = gpt2_model(**inputs, labels=inputs['input_ids']).loss.item()
    return round(math.exp(loss), 2)

print('Computing perplexity...')
ppl_keys = {
    'ppl_orig'          : lambda r: score_sentence_ppl(r['left'], r['span'],                r['right']),
    'ppl_llama'         : lambda r: score_sentence_ppl(r['left'], r['llama_out'],            r['right']),
    'ppl_fim_original'  : lambda r: score_sentence_ppl(r['left'], r['llama_fim_original'],   r['right']),
    'ppl_fim_spm'       : lambda r: score_sentence_ppl(r['left'], r['llama_fim_spm'],        r['right']),
    'ppl_rerank'        : lambda r: score_sentence_ppl(r['left'], r['llama_rerank'],         r['right']),
    'ppl_llada'         : lambda r: score_sentence_ppl(r['left'], r['llada_out'],            r['right']),
}
for key, fn in ppl_keys.items():
    for r in combined_records: r[key] = fn(r)
    mean_ppl = np.mean([r[key] for r in combined_records])
    print(f'  {key:20s}: {mean_ppl:.1f}')

Loading GPT-2 Small as neutral scorer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Computing perplexity...
  ppl_orig            : 26.7
  ppl_llama           : 49.8
  ppl_fim_original    : 62.5
  ppl_fim_spm         : 35.6
  ppl_rerank          : 30.1
  ppl_llada           : 20.1


In [12]:
# ── Summary table ─────────────────────────────────────────────────────────────
bs_means  = {k: round(np.mean(v),3) for k,v in bs_scores.items()}
ppl_means = {k: round(np.mean([r[k] for r in combined_records]),1) for k in ppl_keys}

show_table([
    {'Model / Condition'   :'Original sentence',
     'Conditioning'        :'Ground truth',
     'BERTScore F1 ↑'      :'—',
     'Perplexity ↓'        :ppl_means['ppl_orig']},
    {'Model / Condition'   :'LLaMA-3 left-only',
     'Conditioning'        :'Left context only',
     'BERTScore F1 ↑'      :bs_means['llama_out'],
     'Perplexity ↓'        :ppl_means['ppl_llama']},
    {'Model / Condition'   :'LLaMA-3 FIM original',
     'Conditioning'        :'Instruction prompt',
     'BERTScore F1 ↑'      :bs_means['llama_fim_original'],
     'Perplexity ↓'        :ppl_means['ppl_fim_original']},
    {'Model / Condition'   :'LLaMA-3 FIM SPM',
     'Conditioning'        :'Right context first (Bavarian et al., 2022)',
     'BERTScore F1 ↑'      :bs_means['llama_fim_spm'],
     'Perplexity ↓'        :ppl_means['ppl_fim_spm']},
    {'Model / Condition'   :'LLaMA-3 rerank (N=10)',
     'Conditioning'        :'Best-of-10 by full-sentence score',
     'BERTScore F1 ↑'      :bs_means['llama_rerank'],
     'Perplexity ↓'        :ppl_means['ppl_rerank']},
    {'Model / Condition'   :'LLaDA-8B-Instruct',
     'Conditioning'        :'Full masked sequence (bidirectional)',
     'BERTScore F1 ↑'      :bs_means['llada_out'],
     'Perplexity ↓'        :ppl_means['ppl_llada']},
], title='Contextual Infilling — Results Summary')

# Qualitative examples
show_table([
    {'#':r['id'],'Original span':r['span'][:30]+'...',
     'LLaMA left-only':r['llama_out'][:30],
     'LLaMA rerank'   :r['llama_rerank'][:30],
     'LLaDA'          :r['llada_out'][:30]}
    for r in combined_records if r['id'] in [1,7,10]
], title='Qualitative Examples — Items 1, 7, 10')

# Save and download
with open(RESULTS/'infilling_combined.json','w') as f:
    json.dump({'bs_means':bs_means,'ppl_means':ppl_means,
               'records':combined_records},f,indent=2)
from google.colab import files
files.download(str(RESULTS/'infilling_combined.json'))
print('Downloaded: infilling_combined.json')

Model / Condition,Conditioning,BERTScore F1 ↑,Perplexity ↓
Original sentence,Ground truth,—,26.7
LLaMA-3 left-only,Left context only,0.842,49.8
LLaMA-3 FIM original,Instruction prompt,0.844,62.5
LLaMA-3 FIM SPM,"Right context first (Bavarian et al., 2022)",0.849,35.6
LLaMA-3 rerank (N=10),Best-of-10 by full-sentence score,0.861,30.1
LLaDA-8B-Instruct,Full masked sequence (bidirectional),0.904,20.1


#,Original span,LLaMA left-only,LLaMA rerank,LLaDA
1,the high-powered electron micr...,the microscope. He was looking,the microscope. She zoomed in,a high-powered microscope
7,both parties agreed to a cease...,the House of Representatives p,the leaders of the G20 countri,the two sides reached an agree
10,with complete sincerity and wi...,"“Mommy, I don’t want to go to","""Mommy, I think I want to be a","with a hint of excitement,"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: infilling_combined.json
